<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Data to Decisions: GPU-Accelerated Decision Optimization</b></h1>
<h2><b>Exercise 4:</b> Accelerated QP Solver with cuOpt Python API</h2>
<br>

In this exercise, you will work with the cuOpt Python API for solving Quadratic Programming (QP) models.

> **Student challenge version:** Replace every `<<<TO DO>>>` marker with working Python code, then run the cells in order. The completed reference notebook with the same filename lives one folder up.

> Tip: if a cell raises a syntax error, look for the next `<<<TO DO>>>` marker in that cell.

<hr>

# Portfolio Optimizer for Everyday Investors

You have $10,000 in savings and want to invest across a few familiar assets. How do you balance higher expected return with the risk of large losses?

This notebook builds a simple, realistic portfolio optimization workflow using **NVIDIA cuOpt's QP solver** for quadratic programming. We'll start with a small set of assets, compare equal-weight vs. optimized portfolios, and visualize the efficient frontier.

**Key Concepts:**
- Mean-variance optimization (Markowitz portfolio theory)
- Efficient frontier visualization
- Interactive constraint exploration

References:
- cuOpt LP/QP/MILP API Reference: https://docs.nvidia.com/cuopt/user-guide/latest/cuopt-python/lp-qp-milp/lp-qp-milp-api.html
- Portfolio Optimization: https://en.wikipedia.org/wiki/Portfolio_optimization

## Environment Setup

This notebook uses **NVIDIA cuOpt** for GPU-accelerated quadratic programming optimization.

> **Note:** The QP solver in cuOpt is currently in beta.

Make sure you have cuOpt installed and configured.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cuopt.linear_programming.problem import (
    Problem,
    QuadraticExpression,
    MINIMIZE,
)

%config InlineBackend.figure_format = "retina"

print("Imports ready (using cuOpt QP solver)")


## **Step 1:** Data Setup - A Small, Realistic Universe

We'll work with a compact set of assets an everyday investor might recognize:
- Cash or money market
- US equity ETF proxy
- International equity ETF proxy
- Bond ETF proxy
- Real-asset/REIT or gold ETF proxy

We'll simulate monthly returns with realistic annual return/volatility assumptions and correlations, then estimate the annualized mean returns and covariance matrix.


In [ ]:
# Display the asset assumptions with pandas Styler.
# This requires jinja2 in the Python environment.
summary.style.format({"Annualized Return": "{:.2%}", "Annualized Volatility": "{:.2%}"})


## **Step 2:** Baseline Portfolio - Equal-Weight

Before optimizing, let's compute the equal-weight portfolio. This gives a simple baseline to compare against the optimized portfolios.


In [ ]:
def portfolio_stats(weights, mean_vec, cov_mat):
    exp_return = float(weights @ mean_vec)
    variance = float(weights @ cov_mat @ weights)
    volatility = np.sqrt(variance)
    return exp_return, volatility, variance

n_assets = len(assets)
weights_equal = np.repeat(1.0 / n_assets, n_assets)

ret_eq, vol_eq, var_eq = portfolio_stats(weights_equal, mean_returns, cov_matrix)

baseline_df = pd.DataFrame(
    {
        "Weight": weights_equal,
        "Asset": assets,
    }
).set_index("Asset")

print(f"{ret_eq = }")
print(f"{vol_eq = }")
baseline_df

## **Step 3:** The Optimization Problem

We translate the investor's goals into a quadratic program (QP):

**Decision variables:** Portfolio weights $w_i$ for each asset $i$.

**Objective:** Minimize portfolio variance (risk):

> $\Large\min \frac{1}{2} w^\top \Sigma w$

where $\Sigma$ is the covariance matrix.

**Constraints:**
- Fully invested: $\sum_i w_i = 1$
- Long-only (no shorting): $w_i \geq 0$
- Optional: target minimum return $\mu^\top w \geq r_{target}$
- Optional: max allocation per asset $w_i \leq w_{max}$

### cuOpt QP Implementation

cuOpt's QP solver allows us to express quadratic objectives using `Variable * Variable` syntax.
For the portfolio variance $w^\top \Sigma w$, we construct it as:
> $\Large\sum_i \sum_j w_i \cdot \Sigma_{ij} \cdot w_j$


In [ ]:
def solve_min_variance_qp(
    cov_matrix,
    mean_returns,
    target_return=None,
    max_weight=None,
    min_safe_alloc=None,
    safe_indices=None,
):
    """Solve the minimum-variance portfolio problem using cuOpt QP solver."""
    n = len(mean_returns)
    
    # TODO 04.1: Create the QP model.
    prob = <<<TO DO>>>
    
    upper_bound = max_weight if max_weight is not None else 1.0
    # TODO 04.2: Add one bounded portfolio-weight variable per asset.
    w = [<<<TO DO>>> for i in range(n)]
    
    quad_expr = None
    for i in range(n):
        for j in range(n):
            if abs(cov_matrix[i, j]) > 1e-12:
                if i == 0 and j == 0:
                    # TODO 04.3: Initialize the first risk term.
                    quad_expr = <<<TO DO>>>
                else:
                    # TODO 04.4: Add the remaining risk terms.
                    quad_expr += <<<TO DO>>>
    
    # TODO 04.5: Set the optimization goal.
    prob.<<<TO DO>>>
    
    sum_weights = sum(w)
    # TODO 04.6: Add the fully invested rule.
    prob.<<<TO DO>>>
    
    if target_return is not None:
        expected_return_expr = sum(mean_returns[i] * w[i] for i in range(n))
        # TODO 04.7: Add the target-return rule.
        prob.<<<TO DO>>>
    
    if min_safe_alloc is not None and safe_indices is not None:
        safe_sum = sum(w[i] for i in safe_indices)
        # TODO 04.8: Add the safe-allocation rule.
        prob.<<<TO DO>>>
    
    # TODO 04.9: Solve the QP.
    prob.<<<TO DO>>>
    
    weights = np.array([w[i].Value for i in range(n)])
    portfolio_return = float(mean_returns @ weights)
    portfolio_vol = np.sqrt(float(weights @ cov_matrix @ weights))
    status = "optimal" if prob.Status == 1 else f"status_{prob.Status}"
    return weights, portfolio_return, portfolio_vol, status

print("Solver function defined (using cuOpt QP)")


## **Step 4:** Minimum-Variance Portfolio (No Return Target)

First, let's find the portfolio with the lowest possible risk, without any constraint on expected return. This is the leftmost point on the efficient frontier.


In [ ]:
weights_mv, ret_mv, vol_mv, status_mv = solve_min_variance_qp(cov_matrix, mean_returns)

print(f"Status: {status_mv}")
print(f"\nMinimum-Variance Portfolio:")
print(f"  Expected Return: {ret_mv:.2%}")
print(f"  Volatility:      {vol_mv:.2%}")
print(f"\nWeights:")
for asset, wt in zip(assets, weights_mv):
    print(f"  {asset:<12}: {wt:6.1%}")


## **Step 5:** Efficient Frontier

The efficient frontier shows all portfolios that offer the highest expected return for each level of risk. We trace it by solving QP problems for a range of target returns.


In [ ]:
# Compute efficient frontier by sweeping target returns
min_ret = ret_mv
max_ret = mean_returns.max()
# TODO 04.10: Choose how many points to trace on the frontier. Start with 20.
target_returns = np.linspace(min_ret, max_ret, <<<TO DO>>>)

frontier_vols = []
frontier_rets = []
frontier_weights = []

for target in target_returns:
    w, r, v, _ = solve_min_variance_qp(cov_matrix, mean_returns, target_return=target)
    frontier_vols.append(v)
    frontier_rets.append(r)
    frontier_weights.append(w)

frontier_vols = np.array(frontier_vols)
frontier_rets = np.array(frontier_rets)

print(f"Computed {len(frontier_rets)} points on the efficient frontier.")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Efficient frontier
ax.plot(frontier_vols * 100, frontier_rets * 100, "b-", lw=2, label="Efficient Frontier")

# Individual assets
asset_vols = np.sqrt(np.diag(cov_matrix))
ax.scatter(asset_vols * 100, mean_returns * 100, s=80, c="gray", marker="o", zorder=3, label="Individual Assets")
for i, asset in enumerate(assets):
    ax.annotate(asset, (asset_vols[i] * 100 + 0.3, mean_returns[i] * 100), fontsize=9)

# Equal-weight portfolio
ax.scatter(vol_eq * 100, ret_eq * 100, s=120, c="orange", marker="s", zorder=4, label="Equal-Weight")

# Minimum-variance portfolio
ax.scatter(vol_mv * 100, ret_mv * 100, s=120, c="green", marker="^", zorder=4, label="Min-Variance")

ax.set_xlabel("Volatility (%)")
ax.set_ylabel("Expected Return (%)")
ax.set_title("Efficient Frontier: Risk vs. Return (cuOpt QP)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## **Step 6:** Constrained Portfolio - 5% Target Return + Diversification

Now let's add practical constraints:
- Target at least 5% annual return
- No single asset > 50% (diversification)


In [ ]:
# TODO 04.11: Set a practical target return and allocation cap.
target_5pct = <<<TO DO>>>
max_wt = <<<TO DO>>>

weights_5pct, ret_5pct, vol_5pct, status_5pct = solve_min_variance_qp(
    cov_matrix, mean_returns,
    target_return=target_5pct,
    max_weight=max_wt,
)

print(f"Status: {status_5pct}")
print(f"\nConstrained Portfolio (Target >= 5%, Max Weight 50%):")
print(f"  Expected Return: {ret_5pct:.2%}")
print(f"  Volatility:      {vol_5pct:.2%}")
print(f"\nWeights:")
for asset, wt in zip(assets, weights_5pct):
    print(f"  {asset:<12}: {wt:6.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of weights
x = np.arange(len(assets))
width = 0.35

axes[0].bar(x - width/2, weights_equal * 100, width, label="Equal-Weight", color="orange")
axes[0].bar(x + width/2, weights_5pct * 100, width, label="Optimized (5%+ target)", color="steelblue")
axes[0].set_ylabel("Weight (%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(assets, rotation=15)
axes[0].legend()
axes[0].set_title("Portfolio Allocation Comparison")

# Scatter: return vs vol
axes[1].scatter(vol_eq * 100, ret_eq * 100, s=150, c="orange", marker="s", label=f"Equal-Weight ({ret_eq:.1%}, {vol_eq:.1%})")
axes[1].scatter(vol_5pct * 100, ret_5pct * 100, s=150, c="steelblue", marker="^", label=f"Optimized ({ret_5pct:.1%}, {vol_5pct:.1%})")
axes[1].plot(frontier_vols * 100, frontier_rets * 100, "k--", alpha=0.4, label="Frontier")
axes[1].set_xlabel("Volatility (%)")
axes[1].set_ylabel("Expected Return (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[1].set_title("Return vs. Volatility")

plt.tight_layout()
plt.show()


## **Step 7:** Interactive Exploration - Adjust Your Preferences

Now let's explore how changing your preferences affects the optimal portfolio. You can adjust:
- **Target return**: How much growth do you want?
- **Maximum weight per asset**: Avoid over-concentration
- **Minimum safe allocation**: Keep a floor in Cash + Bonds


In [ ]:
# Interactive exploration function
# In Jupyter, uncomment the interact() call at the bottom to enable sliders

def explore_portfolio(target_return_pct=5.0, max_weight_pct=60.0, min_safe_pct=10.0):
    """Explore portfolio with given parameters."""
    target_return = target_return_pct / 100.0
    max_weight = max_weight_pct / 100.0
    min_safe = min_safe_pct / 100.0
    safe_indices = [0, 3]  # Cash and Bond
    
    w, r, v, status = solve_min_variance_qp(
        cov_matrix, mean_returns,
        target_return=target_return,
        max_weight=max_weight,
        min_safe_alloc=min_safe,
        safe_indices=safe_indices,
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Weights
    colors = ["#2ecc71" if i in safe_indices else "#3498db" for i in range(len(assets))]
    axes[0].barh(assets, w * 100, color=colors)
    axes[0].set_xlabel("Weight (%)")
    axes[0].set_xlim(0, 70)
    axes[0].set_title("Optimal Allocation")
    axes[0].axvline(max_weight * 100, color="red", linestyle="--", label=f"Max {max_weight_pct:.0f}%")
    axes[0].legend()
    
    # Frontier with current point
    axes[1].plot(frontier_vols * 100, frontier_rets * 100, "b-", lw=2, alpha=0.5)
    axes[1].scatter(v * 100, r * 100, s=200, c="red", marker="*", zorder=5, label="Your Portfolio")
    axes[1].scatter(vol_eq * 100, ret_eq * 100, s=80, c="orange", marker="s", zorder=4, label="Equal-Weight")
    axes[1].set_xlabel("Volatility (%)")
    axes[1].set_ylabel("Expected Return (%)")
    axes[1].set_title("Your Portfolio on the Frontier")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Status: {status}")
    print(f"Expected Return: {r:.2%}  |  Volatility: {v:.2%}")
    print(f"Safe Assets (Cash + Bond): {(w[0] + w[3]):.1%}")

# Demo with default parameters (static execution)
explore_portfolio(target_return_pct=5.0, max_weight_pct=60.0, min_safe_pct=10.0)

# To enable interactive sliders in Jupyter, uncomment:
# from ipywidgets import interact, FloatSlider
# interact(
#     explore_portfolio,
#     target_return_pct=FloatSlider(value=5.0, min=2.0, max=8.0, step=0.5, description="Target Return %"),
#     max_weight_pct=FloatSlider(value=60.0, min=25.0, max=100.0, step=5.0, description="Max Weight %"),
#     min_safe_pct=FloatSlider(value=10.0, min=0.0, max=50.0, step=5.0, description="Min Safe %"),
# )


## **Step 8:** Interpretation - What It Means for You

Let's put one scenario in plain English. Suppose you target a 6% annual return with at least 20% in safe assets.


In [ ]:
target_scenario = 0.06
min_safe_scenario = 0.20
safe_indices = [0, 3]

w_scenario, r_scenario, v_scenario, _ = solve_min_variance_qp(
    cov_matrix, mean_returns,
    target_return=target_scenario,
    min_safe_alloc=min_safe_scenario,
    safe_indices=safe_indices,
)

print("=" * 60)
print("YOUR RECOMMENDED PORTFOLIO")
print("=" * 60)
print(f"\nTarget: At least {target_scenario:.0%} annual return")
print(f"Constraint: Keep at least {min_safe_scenario:.0%} in Cash + Bonds\n")

for asset, wt in zip(assets, w_scenario):
    bar = "█" * int(wt * 40)
    print(f"  {asset:<12} {wt:5.1%}  {bar}")

print(f"\n→ Expected Return: {r_scenario:.2%}")
print(f"→ Volatility:      {v_scenario:.2%}")
print(f"→ Safe Allocation: {(w_scenario[0] + w_scenario[3]):.1%}")

print("\n" + "=" * 60)
print("WHAT THIS MEANS")
print("=" * 60)
print(f"""
If you invest $10,000 according to this allocation:
  - Cash:        ${10000 * w_scenario[0]:,.0f}
  - US Equity:   ${10000 * w_scenario[1]:,.0f}
  - Intl Equity: ${10000 * w_scenario[2]:,.0f}
  - Bonds:       ${10000 * w_scenario[3]:,.0f}
  - REIT/Gold:   ${10000 * w_scenario[4]:,.0f}

You can expect ~{r_scenario:.1%} growth per year on average,
with a typical annual swing of about +/-{v_scenario:.1%}.
""")


## **Step 9:** Summary & Next Steps

### What We Covered

1. **The Trade-off**: Higher expected return requires accepting more volatility (risk).
2. **Efficient Frontier**: Shows the best possible risk-return combinations.
3. **QP Formulation**: Minimizing portfolio variance subject to constraints is a quadratic program.
4. **cuOpt QP Solver**: GPU-accelerated QP solver for fast optimization.
5. **Personalization**: Constraints like target return, max weight, and min safe allocation let you tailor the solution.

### cuOpt QP Quick Reference for Portfolio Optimization

```python
from cuopt.linear_programming.problem import Problem, QuadraticExpression, MINIMIZE
import numpy as np

# Create problem
prob = Problem("Portfolio")

# Decision variables with bounds
n = len(mean_returns)
w = [prob.addVariable(lb=0.0, ub=1.0, name=f"w_{i}") for i in range(n)]

# Quadratic objective: minimize w' * cov_matrix * w
quad_expr = None
for i in range(n):
    for j in range(n):
        if abs(cov_matrix[i, j]) > 1e-12:
            if(i==0 and j==0):
                quad_expr = float(cov_matrix[i, j]) * w[i] * w[j]
            else:
                quad_expr += float(cov_matrix[i, j]) * w[i] * w[j]

prob.setObjective(quad_expr, sense=MINIMIZE)

# Constraints
prob.addConstraint(sum(w) == 1)                               # Fully invested
prob.addConstraint(sum(mean_returns[i] * w[i] for i in range(n)) >= target)  # Min return

# Solve
prob.solve()
weights = np.array([w[i].Value for i in range(n)])
```

### Possible Extensions

- **Real data**: Replace simulated returns with actual ETF price history (e.g., via `yfinance`)
- **Transaction costs**: Penalize turnover from a current portfolio
- **Cardinality constraints**: Limit the number of assets held (MIQP)
- **Risk parity**: Equal risk contribution from each asset
- **Robust optimization**: Account for uncertainty in return estimates
- **Large-scale problems**: cuOpt is designed for GPU acceleration on large problems

### Resources

- [cuOpt LP/QP/MILP API Reference](https://docs.nvidia.com/cuopt/user-guide/latest/cuopt-python/lp-qp-milp/lp-qp-milp-api.html)
- [Financial Portfolio Optimization](https://developer.nvidia.com/blog/accelerating-real-time-financial-decisions-with-quantitative-portfolio-optimization/)
- [cuOpt QP Example](https://docs.nvidia.com/cuopt/user-guide/latest/_downloads/92b89a8b87f10add8b883b151de613cb/simple_qp_example.py)
- [Financial Portfolio Optimization Notebooks](https://github.com/NVIDIA-AI-Blueprints/quantitative-portfolio-optimization)
- [Portfolio Optimization (Wikipedia)](https://en.wikipedia.org/wiki/Portfolio_optimization)
- [Modern Portfolio Theory](https://www.investopedia.com/terms/m/modernportfoliotheory.asp)


**Congratulations!** You finished this exercise by solving portfolio optimization problems with cuOpt QP solver.

**In the last exercise, you will build an end-to-end agentic workflow that uses natural language together with cuOpt for decision optimization.**

<img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>